# Data Changes During Regression Stage

This notebook documents all changes made to files in `data/clean/` during the regression
analysis stage of the project. It serves as an audit trail so any downstream notebook can
understand what data was modified, why, and what was deliberately left unchanged.

## Overview

| # | File changed | Type of change | Rows before | Rows after |
|---|-------------|----------------|------------|------------|
| 1 | `data/clean/uk_grouped/with_categorykey/`<br>`UK_enrollments_grouped_comparison_all_years_with_categorykey.csv` | 12 rows appended | 282 | 294 |
| 2 | `data/clean/uk_grouped/with_categorykey/`<br>`UK_enrollments_grouped_comparison_all_years_with_categorykey.csv` | 3 rows appended | 294 | 297 |
| 3 | `data/clean/uk_grouped/with_categorykey/`<br>`UK_enrollments_grouped_comparison_all_years_with_categorykey.csv` | 3 rows appended | 297 | 300 |

**Files deliberately NOT changed:**
- `data/clean/uk_grouped/with_categorykey/with_category_name/UK_enrollments_grouped_comparison_all_years_with_category.csv` — used by EDA notebooks; left unchanged to avoid affecting completed EDA analyses
- `data/clean/uk_grouped/with_categorykey/UK_enrollments_grouped_2016-17_with_categorykey.csv`
- `data/clean/uk_grouped/with_categorykey/UK_enrollments_grouped_2017-18_with_categorykey.csv`
- `data/clean/uk_grouped/with_categorykey/UK_enrollments_grouped_2018-19_with_categorykey.csv`

> **Note:** The three individual year files above contain incorrect category key mappings for
> some subjects. These errors were not corrected at this stage as they are out of scope for the
> current regression work. They should be addressed in a future data cleaning pass.


## Change 1 — Extend UK comparison file to include 2016/17–2018/19

### Why this change was made

The four REG notebooks (Architecture & Building, Creative Arts, Education, Engineering &
Related Tech) built their DiD panels starting from 2019 because the UK comparison file only
contained data from 2019/20 onwards for category keys 3, 4, 7, and 10.

Investigation showed that raw UK HESA data for 2016/17–2018/19 *does* exist in the individual
year files, but uses an older JACS (Joint Academic Coding System) subject taxonomy rather than
the CAH (Common Aggregation Hierarchy) taxonomy used from 2019/20 onwards. The earlier files had
not been remapped to the AUS category key scheme for the four disciplines of interest.

Extending the panel to 2016 adds **5 pre-treatment years** (2016–2020) instead of 2, enabling a
richer parallel trends test and improving degrees of freedom from df=4 to df=7 in each notebook.

### JACS subject → AUS category key mapping applied

| JACS Subject (2016–2018 files) | AUS Category Key | AUS Category Name |
|-------------------------------|-----------------|-------------------|
| `09 Engineering & technology` | 3 | Engineering & Related Tech |
| `(A) Architecture, building & planning` | 4 | Architecture & Building |
| `(I) Education` | 7 | Education |
| `(H) Creative arts & design` | 10 | Creative Arts |

Each of these subjects appears as a single row in the source files, so no aggregation was
required — the `Total UK` value is taken directly from the source file.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data').exists():
    ROOT = ROOT.parent

comp_path = ROOT / 'data/clean/uk_grouped/with_categorykey/UK_enrollments_grouped_comparison_all_years_with_categorykey.csv'
df = pd.read_csv(comp_path)

added = df[
    df['AcademicYear'].isin(['2016/17', '2017/18', '2018/19']) &
    df['categorykey'].isin([3, 4, 7, 10])
].copy()

KEY_NAME = {3: 'Engineering & Related Tech', 4: 'Architecture & Building',
            7: 'Education', 10: 'Creative Arts'}
added['Category'] = added['categorykey'].map(KEY_NAME)

print('=== 12 rows appended to comparison file ===')
display(added[['AcademicYear', 'categorykey', 'Category', 'Subject', 'Total UK']]
        .sort_values(['AcademicYear', 'categorykey'])
        .reset_index(drop=True))

=== 12 rows appended to comparison file ===


,AcademicYear,categorykey,Category,Subject,Total UK
0,2016/17,3,Engineering & Related Tech,09 Engineering & technology,165155.0
1,2016/17,4,Architecture & Building,"(A) Architecture, building & planning",51265.0
2,2016/17,7,Education,(I) Education,151060.0
3,2016/17,10,Creative Arts,(H) Creative arts & design,175700.0
4,2017/18,3,Engineering & Related Tech,09 Engineering & technology,164975.0
5,2017/18,4,Architecture & Building,"(A) Architecture, building & planning",53620.0
6,2017/18,7,Education,(I) Education,145445.0
7,2017/18,10,Creative Arts,(H) Creative arts & design,178415.0
8,2018/19,3,Engineering & Related Tech,09 Engineering & technology,165180.0
9,2018/19,4,Architecture & Building,"(A) Architecture, building & planning",55345.0


In [2]:
# Verify each row against its source individual-year file
source_files = {
    '2016/17': ROOT / 'data/clean/uk_grouped/with_categorykey/UK_enrollments_grouped_2016-17_with_categorykey.csv',
    '2017/18': ROOT / 'data/clean/uk_grouped/with_categorykey/UK_enrollments_grouped_2017-18_with_categorykey.csv',
    '2018/19': ROOT / 'data/clean/uk_grouped/with_categorykey/UK_enrollments_grouped_2018-19_with_categorykey.csv',
}

JACS_SUBJECTS = [
    '09 Engineering & technology',
    '(A) Architecture, building & planning',
    '(I) Education',
    '(H) Creative arts & design',
]

print('=== Source verification: Total UK values match original files ===\n')
all_ok = True
for acad_year, fpath in source_files.items():
    src = pd.read_csv(fpath)
    for subj in JACS_SUBJECTS:
        src_row  = src[src['Subject'] == subj]
        comp_row = added[(added['AcademicYear'] == acad_year) & (added['Subject'] == subj)]
        if src_row.empty or comp_row.empty:
            print(f'  MISSING: {acad_year} | {subj}')
            all_ok = False
            continue
        src_val  = src_row['Total UK'].iloc[0]
        comp_val = comp_row['Total UK'].iloc[0]
        match = abs(src_val - comp_val) < 1
        status = 'OK' if match else 'MISMATCH'
        if not match:
            all_ok = False
        print(f'  {status}  {acad_year} | {subj[:40]:40s} | source={src_val:,.0f}  comp={comp_val:,.0f}')

print()
print('All values verified.' if all_ok else 'WARNING: mismatches found above.')

=== Source verification: Total UK values match original files ===

  OK  2016/17 | 09 Engineering & technology              | source=165,155  comp=165,155
  OK  2016/17 | (A) Architecture, building & planning    | source=51,265  comp=51,265
  OK  2016/17 | (I) Education                            | source=151,060  comp=151,060
  OK  2016/17 | (H) Creative arts & design               | source=175,700  comp=175,700
  OK  2017/18 | 09 Engineering & technology              | source=164,975  comp=164,975
  OK  2017/18 | (A) Architecture, building & planning    | source=53,620  comp=53,620
  OK  2017/18 | (I) Education                            | source=145,445  comp=145,445
  OK  2017/18 | (H) Creative arts & design               | source=178,415  comp=178,415
  OK  2018/19 | 09 Engineering & technology              | source=165,180  comp=165,180
  OK  2018/19 | (A) Architecture, building & planning    | source=55,345  comp=55,345
  OK  2018/19 | (I) Education                            | 

In [3]:
print('=== Comparison file row counts by AcademicYear ===')
year_counts = df.groupby('AcademicYear').size().rename('rows')
print(year_counts.to_string())
print(f'\nTotal rows: {len(df)} (was 282 before; 294 after Change 1; 297 after Change 2; 300 after Change 3)')

=== Comparison file row counts by AcademicYear ===
AcademicYear
2013/14    22
2014/15    22
2015/16    22
2016/17    28
2017/18    28
2018/19    28
2019/20    25
2020/21    25
2021/22    25
2022/23    25
2023/24    25
2024/25    25

Total rows: 300 (was 282 before; 294 after Change 1; 297 after Change 2; 300 after Change 3)


## Change 2 — Extend UK comparison file to include IT (2016/17–2018/19)

### Why this change was made

During the REG Information Technology regression analysis (CategoryKey = 2), the UK DiD panel
was initially restricted to 2019–2024 (N = 12, df = 4) because the comparison file contained
no pre-2019 rows for CategoryKey 2.

Investigation of the individual year source files confirmed that **UK IT data does exist** for
2016/17–2018/19 under the JACS subject '08 Computer science'. However, the individual year
files mapped this subject to categorykey = 11 (Others) rather than categorykey = 2 (Information
Technology). This subject was omitted when the comparison file was extended during Change 1 —
which covered Architecture (key 4), Creative Arts (key 10), Education (key 7), and Engineering
(key 3) — but did not include IT.

Three rows were appended to the comparison file with categorykey = 2, enabling a full
2016–2024 panel (N = 18, df = 7) for the IT DiD analysis.

### JACS subject → AUS category key mapping applied

| JACS Subject (2016–2018 files) | AUS Category Key | AUS Category Name |
|-------------------------------|-----------------|-------------------|
| `08 Computer science` | 2 | Information Technology |

The `Total UK` values come directly from the JACS source files (single row per year — no
aggregation required).


In [4]:
added2 = df[
    df['AcademicYear'].isin(['2016/17', '2017/18', '2018/19']) &
    (df['categorykey'] == 2)
].copy()
added2['Category'] = 'Information Technology'

print('=== 3 rows appended to comparison file (Change 2) ===')
display(added2[['AcademicYear', 'categorykey', 'Category', 'Subject', 'Total UK']]
        .sort_values('AcademicYear')
        .reset_index(drop=True))

# Verify against source files
source_files = {
    '2016/17': ROOT / 'data/clean/uk_grouped/with_categorykey/UK_enrollments_grouped_2016-17_with_categorykey.csv',
    '2017/18': ROOT / 'data/clean/uk_grouped/with_categorykey/UK_enrollments_grouped_2017-18_with_categorykey.csv',
    '2018/19': ROOT / 'data/clean/uk_grouped/with_categorykey/UK_enrollments_grouped_2018-19_with_categorykey.csv',
}

print()
print('=== Source verification ===')
all_ok = True
for acad_year, fpath in source_files.items():
    src = pd.read_csv(fpath)
    src_row  = src[src['Subject'] == '08 Computer science']
    comp_row = added2[added2['AcademicYear'] == acad_year]
    if src_row.empty or comp_row.empty:
        print(f'  MISSING: {acad_year} | 08 Computer science')
        all_ok = False
        continue
    src_val  = src_row['Total UK'].iloc[0]
    comp_val = comp_row['Total UK'].iloc[0]
    match    = abs(src_val - comp_val) < 1
    if not match:
        all_ok = False
    print(f'  {"OK" if match else "MISMATCH"}  {acad_year} | 08 Computer science | '
          f'source={src_val:,.0f}  comp={comp_val:,.0f}')

print()
print('All values verified.' if all_ok else 'WARNING: mismatches found.')


=== 3 rows appended to comparison file (Change 2) ===


,AcademicYear,categorykey,Category,Subject,Total UK
0,2016/17,2,Information Technology,08 Computer science,101145.0
1,2017/18,2,Information Technology,08 Computer science,107250.0
2,2018/19,2,Information Technology,08 Computer science,114730.0



=== Source verification ===
  OK  2016/17 | 08 Computer science | source=101,145  comp=101,145
  OK  2017/18 | 08 Computer science | source=107,250  comp=107,250
  OK  2018/19 | 08 Computer science | source=114,730  comp=114,730

All values verified.


## Change 3 — Extend UK comparison file to include M&C (2016/17–2018/19)

### Why this change was made

During the REG Management & Commerce regression analysis (CategoryKey = 8), the UK DiD panel
would have been restricted to 2019–2024 because the comparison file contained no pre-2019 rows
for CategoryKey 8.

Investigation of the individual year source files confirmed that **UK M&C data exists** for
2016/17–2018/19 under the JACS subject '(D) Business & administrative studies'. The individual
year files mapped this subject to categorykey = 11 (Others) rather than categorykey = 8
(Management & Commerce) — the same misassignment pattern found in Changes 1 and 2.

Three rows were appended to the comparison file with categorykey = 8, enabling a full
2016–2024 panel (N = 18, df = 7) for the M&C DiD analysis.

### JACS subject → AUS category key mapping applied

| JACS Subject (2016–2018 files) | AUS Category Key | AUS Category Name |
|-------------------------------|-----------------|-------------------|
| `(D) Business & administrative studies` | 8 | Management & Commerce |

The `Total UK` values come directly from the JACS source files (single row per year — no
aggregation required).


In [5]:
added3 = df[
    df['AcademicYear'].isin(['2016/17', '2017/18', '2018/19']) &
    (df['categorykey'] == 8)
].copy()
added3['Category'] = 'Management & Commerce'

print('=== 3 rows appended to comparison file (Change 3) ===')
display(added3[['AcademicYear', 'categorykey', 'Category', 'Subject', 'Total UK']]
        .sort_values('AcademicYear')
        .reset_index(drop=True))

# Verify against source files
print()
print('=== Source verification ===')
all_ok = True
for acad_year, fpath in source_files.items():
    src = pd.read_csv(fpath)
    src_row  = src[src['Subject'] == '(D) Business & administrative studies']
    comp_row = added3[added3['AcademicYear'] == acad_year]
    if src_row.empty or comp_row.empty:
        print(f'  MISSING: {acad_year} | (D) Business & administrative studies')
        all_ok = False
        continue
    src_val  = src_row['Total UK'].iloc[0]
    comp_val = comp_row['Total UK'].iloc[0]
    match    = abs(src_val - comp_val) < 1
    if not match:
        all_ok = False
    print(f'  {"OK" if match else "MISMATCH"}  {acad_year} | (D) Business & administrative studies | '
          f'source={src_val:,.0f}  comp={comp_val:,.0f}')

print()
print('All values verified.' if all_ok else 'WARNING: mismatches found.')


=== 3 rows appended to comparison file (Change 3) ===


,AcademicYear,categorykey,Category,Subject,Total UK
0,2016/17,8,Management & Commerce,(D) Business & administrative studies,333425.0
1,2017/18,8,Management & Commerce,(D) Business & administrative studies,342970.0
2,2018/19,8,Management & Commerce,(D) Business & administrative studies,358480.0



=== Source verification ===
  OK  2016/17 | (D) Business & administrative studies | source=333,425  comp=333,425
  OK  2017/18 | (D) Business & administrative studies | source=342,970  comp=342,970
  OK  2018/19 | (D) Business & administrative studies | source=358,480  comp=358,480

All values verified.


## Impact on REG Notebooks

The following notebooks were updated following this data change. Changes were purely in code and
markdown (no further data file modifications):

| Notebook | Change |
|----------|--------|
| `REG Architecture & Building.ipynb` | DiD panel extended 2019→2016; markdown updated (N=18, df=7); re-executed |
| `REG Creative Arts.ipynb` | DiD panel extended 2019→2016; markdown updated (N=18, df=7); re-executed |
| `REG Education.ipynb` | DiD panel extended 2019→2016; markdown updated (N=18, df=7); re-executed |
| `REG Engineering & Related Tech.ipynb` | DiD panel extended 2019→2016; markdown updated (N=18, df=7); re-executed |
| `REG Information Technology.ipynb` | DiD panel extended 2019→2016 (Change 2 data); markdown updated (N=18, df=7); re-executed |
| `REG Management & Commerce.ipynb` | DiD panel extended 2019→2016 (Change 3 data); markdown updated (N=18, df=7); re-executed |

### Key result changes from panel extension

| Discipline | β_did (N=12, 2019–2024) | p-value | β_did (N=18, 2016–2024) | p-value |
|------------|------------------------|---------|------------------------|---------|
| Architecture & Building | −9.7% | 0.038 ✓ | −6.1% | 0.279 |
| Creative Arts | −0.6% | 0.858 | −2.2% | 0.471 |
| Education | +16.8% | 0.166 | +25.7% | 0.059 ✓ (marginal) |
| Engineering & Related Tech | −5.5% | 0.558 | −7.8% | 0.227 |
| Information Technology | −12.8% | 0.305 | −7.3% | 0.483 |
| Management & Commerce | n/a (no prior restricted panel) | — | −39.4% | <0.001 ✓✓✓ |

The extended pre-period also revealed pre-trend concerns invisible with only one pre-period.
See individual REG notebooks (Results Summary cells) for discipline-specific notes.

### Known issue in pre-2019 individual year files (not corrected)

The `with_categorykey` files for 2016/17–2018/19 contain incorrect mappings for several other
subjects beyond the four added above. For reference:

| JACS Subject | Existing (wrong) key | Correct key | Category name |
|-------------|---------------------|-------------|---------------|
| `06 Physical sciences` | 5 | 1 | Natural & Physical Science |
| `09 Engineering & technology` | 1 | 3 | Engineering & Related Tech |
| `05 Agriculture & related subjects` | 6 | 5 | Environment & Related |
| `04 Veterinary science` | 9 | 6 | Health |
| `(A) Architecture, building & planning` | 11 | 4 | Architecture & Building |
| `(I) Education` | 11 | 7 | Education |
| `(B) Social studies` | 11 | 9 | Society & Culture |
| `(C) Law` | 11 | 9 | Society & Culture |
| `(E) Mass communications & documentation` | 11 | 9 | Society & Culture |
| `(F) Languages` | 11 | 9 | Society & Culture |
| `(G) Historical & philosophical studies` | 11 | 9 | Society & Culture |
| `(H) Creative arts & design` | 11 | 10 | Creative Arts |

These errors do not affect the REG notebooks (which use the comparison file, not the individual
year files), but should be corrected in a future data cleaning pass if the individual year files
are ever used directly.